# 🛡️ External Authorization with Open Policy Agent (OPA) in Apache Polaris

This notebook demonstrates how to configure **Apache Polaris v1.3.0** to delegate authorization decisions to an external **Open Policy Agent (OPA)** engine.

By default, Polaris uses built-in Role-Based Access Control (RBAC). When OPA is enabled (via `polaris.authorization.type=opa`), Polaris forwards every action attempt to an OPA endpoint. OPA evaluates the request against our custom Rego policy and returns `allow: true` or `allow: false`.

## 🎭 Roles Defined in our Rego Policy
- **`ADMIN`**: Can do everything (Create catalogs, namespaces, tables, list them etc.)
- **`DATA_ENGINEER`**: Can create, update and read namespaces and tables, but cannot manage Catalogs.
- **`ANALYST`**: Can only perform read-only operations (`LOAD_TABLE_WITH_READ_DELEGATION`, `LIST_NAMESPACES`, `LIST_TABLES`, `LIST_CATALOGS`).

---
## 🛠️ Step 1 — Environment Setup

In [1]:
import requests
import json
import time

# --- Polaris (Management API is port 8181) ---
POLARIS_URL = "http://polaris:8181"
POLARIS_CLIENT_ID = "root"
POLARIS_CLIENT_SECRET = "polaris-secret"

# --- OPA (Internal network port 8181) ---
OPA_URL = "http://opa:8181"

def wait_for_service(name, url, max_attempts=30):
    print(f"⏳ Waiting for {name}...", end="")
    for _ in range(max_attempts):
        try:
            r = requests.get(url, timeout=2)
            print(f" ✅ {name} ready! (HTTP {r.status_code})")
            return True
        except:
            print(".", end="", flush=True)
            time.sleep(2)
    print(f" ❌ {name} failed to become ready.")
    return False

wait_for_service("Polaris REST API", f"{POLARIS_URL}/api/catalog/v1/config")
wait_for_service("OPA Server", f"{OPA_URL}/health")

⏳ Waiting for Polaris REST API... ✅ Polaris REST API ready! (HTTP 401)
⏳ Waiting for OPA Server... ✅ OPA Server ready! (HTTP 200)


True

### Inspect the loaded OPA Rego Policy
Let's query the OPA API directly to view the policy that we mounted via Docker.

In [2]:
r = requests.get(f"{OPA_URL}/v1/policies/polaris_authz")
if r.ok:
    print("✅ Polaris OPA Policy loaded. Source:\n")
    print(r.json()["result"]["raw"])
else:
    print(f"❌ Failed to fetch policy: {r.status_code}")
    print(r.text)

❌ Failed to fetch policy: 404
{
  "code": "resource_not_found",
  "message": "storage_not_found_error: policy id \"polaris_authz\""
}



---
## 🔑 Step 2 — Create Test Principals and Roles
We need to create the users internally in Polaris and attach them to the roles that our OPA policy checks (`ADMIN`, `DATA_ENGINEER`, `ANALYST`).

We will do this using the `root` admin token.

In [3]:
def get_token(client_id, client_secret):
    resp = requests.post(
        f"{POLARIS_URL}/api/catalog/v1/oauth/tokens",
        data={"grant_type": "client_credentials", "client_id": client_id, "client_secret": client_secret, "scope": "PRINCIPAL_ROLE:ALL"}
    )
    resp.raise_for_status()
    return resp.json()['access_token']

root_token = get_token(POLARIS_CLIENT_ID, POLARIS_CLIENT_SECRET)
root_headers = {"Authorization": f"Bearer {root_token}", "Content-Type": "application/json"}
print("✅ Obtained root token")

def setup_principal(name, role_name):
    """Creates a principal, their role, and grants the role to the principal via the Polaris Mgmt API"""
    # 1. Create Principal
    p_resp = requests.post(
        f"{POLARIS_URL}/api/management/v1/principals", headers=root_headers,
        json={"principal": {"name": name, "type": "SERVICE"}}
    )
    if p_resp.status_code == 409:
        # Already exists, fetch it and recreate credentials by rotating
        p_creds = requests.put(f"{POLARIS_URL}/api/management/v1/principals/{name}/credentials", headers=root_headers).json()
    else:
        p_resp.raise_for_status()
        p_creds = p_resp.json()
        
    # 2. Create Principal Role
    r_resp = requests.post(
        f"{POLARIS_URL}/api/management/v1/principal-roles", headers=root_headers,
        json={"principalRole": {"name": role_name}}
    )
    if r_resp.status_code != 409:
        r_resp.raise_for_status()

    # 3. Grant Role to Principal
    requests.put(
        f"{POLARIS_URL}/api/management/v1/principals/{name}/principal-roles", headers=root_headers,
        json={"role": {"name": role_name}, "privilege": "PRINCIPAL_ROLE_USAGE"}
    )
    
    client_id = p_creds.get('credentials', p_creds).get('clientId')
    client_secret = p_creds.get('credentials', p_creds).get('clientSecret')
    print(f"✅ Configured principal: '{name}' assigned to role '{role_name}'")
    return get_token(client_id, client_secret)

# Create matching principals and roles
token_admin = setup_principal("polaris-admin", "ADMIN")
token_engineer = setup_principal("polaris-data-engineer", "DATA_ENGINEER")
token_analyst = setup_principal("polaris-analyst", "ANALYST")

✅ Obtained root token
✅ Configured principal: 'polaris-admin' assigned to role 'ADMIN'
✅ Configured principal: 'polaris-data-engineer' assigned to role 'DATA_ENGINEER'
✅ Configured principal: 'polaris-analyst' assigned to role 'ANALYST'


---
## 👮‍♂️ Scenario 1: Admin User
The admin user (`ADMIN` role) can do anything. Let's create a catalog, a namespace, and a table.

In [4]:
admin_headers = {"Authorization": f"Bearer {token_admin}", "Content-Type": "application/json"}

# 1. Create Catalog
cat_payload = {
    "catalog": {
        "name": "opa_catalog", "type": "INTERNAL",
        "properties": {"default-base-location": "s3://lakehouse/opa_catalog"},
        "storageConfigInfo": {
            "storageType": "S3", "stsUnavailable": True, "pathStyleAccess": True,
            "endpoint": "http://seaweedfs:8333", "region": "us-east-1",
            "allowedLocations": ["s3://lakehouse/"]
        }
    }
}
r = requests.post(f"{POLARIS_URL}/api/management/v1/catalogs", headers=admin_headers, json=cat_payload)
if r.status_code in (200, 201):
    print("✅ ADMIN: Successfully created catalog 'opa_catalog' (HTTP 200)")
elif r.status_code == 409:
    print("✅ ADMIN: Catalog 'opa_catalog' already exists (HTTP 409)")
else:
    print(f"❌ ADMIN: Error creating catalog: {r.status_code} - {r.text}")

# 2. Create Namespace
r = requests.post(
    f"{POLARIS_URL}/api/catalog/v1/opa_catalog/namespaces", headers=admin_headers,
    json={"namespace": ["finance"], "properties": {}}
)
if r.status_code in (200, 201) or r.status_code == 409:
    print("✅ ADMIN: Successfully managed namespace 'finance'")
else:
    print(f"❌ ADMIN: Error with namespace: {r.status_code} - {r.text}")

❌ ADMIN: Error creating catalog: 403 - {"error":{"message":"OPA denied authorization","type":"ForbiddenException","code":403}}
❌ ADMIN: Error with namespace: 404 - {"error":{"message":"Namespace does not exist: ","type":"NoSuchNamespaceException","code":404}}


---
## 🧑‍💻 Scenario 2: Data Engineer User
The Data Engineer (`DATA_ENGINEER` role) is allowed to read and make changes to tables/namespaces, but **cannot create new catalogs**.

In [5]:
engineer_headers = {"Authorization": f"Bearer {token_engineer}", "Content-Type": "application/json"}

# ❌ 1. Try to create a new Catalog (Should fail according to Rego policy)
cat_payload["catalog"]["name"] = "engineer_catalog"
r_cat = requests.post(f"{POLARIS_URL}/api/management/v1/catalogs", headers=engineer_headers, json=cat_payload)
if r_cat.status_code == 403:
    print(f"✅ DATA ENGINEER: Correctly blocked from creating a catalog (HTTP 403 Forbidden)")
else:
    print(f"❌ Unexpected result: {r_cat.status_code} - {r_cat.text}")

# ✅ 2. Create a Table inside an existing Namespace (Should succeed)
table_payload = {
    "name": "transactions",
    "schema": {"type": "struct", "schema-id": 0, "fields": [{"id": 1, "name": "amount", "required": True, "type": "double"}]},
    "properties": {},
    "partition-spec": {"spec-id": 0, "fields": []}
}
r_tab = requests.post(
    f"{POLARIS_URL}/api/catalog/v1/opa_catalog/namespaces/finance/tables",
    headers=engineer_headers, json=table_payload
)
if r_tab.status_code in (200, 201) or r_tab.status_code == 409:
    print("✅ DATA ENGINEER: Successfully managed table 'transactions'")
else:
    print(f"❌ Error creating table: {r_tab.status_code} - {r_tab.text}")

✅ DATA ENGINEER: Correctly blocked from creating a catalog (HTTP 403 Forbidden)
❌ Error creating table: 404 - {"error":{"message":"Namespace does not exist: finance","type":"NoSuchNamespaceException","code":404}}


---
## 📈 Scenario 3: Analyst User
The Analyst (`ANALYST` role) is an observer. They can load table metadata (`LOAD_TABLE_WITH_READ_DELEGATION`) but **cannot create tables**.

In [6]:
analyst_headers = {"Authorization": f"Bearer {token_analyst}", "Content-Type": "application/json"}

# ✅ 1. Read Table Metadata (Should succeed)
r_read = requests.get(f"{POLARIS_URL}/api/catalog/v1/opa_catalog/namespaces/finance/tables/transactions", headers=analyst_headers)
if r_read.status_code == 200:
    print("✅ ANALYST: Successfully loaded 'transactions' table metadata")
else:
    print(f"❌ Unsuccessful read: {r_read.status_code} - {r_read.text}")

# ❌ 2. Try to create a Table (Should fail)
table_payload["name"] = "forecasts"
r_write = requests.post(
    f"{POLARIS_URL}/api/catalog/v1/opa_catalog/namespaces/finance/tables",
    headers=analyst_headers, json=table_payload
)
if r_write.status_code == 403:
    print("✅ ANALYST: Correctly blocked from creating 'forecasts' table (HTTP 403 Forbidden)")
else:
    print(f"❌ Unexpected result: {r_write.status_code} - {r_write.text}")

❌ Unsuccessful read: 404 - {"error":{"message":"Table does not exist: finance.transactions","type":"NoSuchTableException","code":404}}
❌ Unexpected result: 404 - {"error":{"message":"Namespace does not exist: finance","type":"NoSuchNamespaceException","code":404}}


---
## 🕵️‍♂️ Under the Hood: The OPA REST API
How does OPA actually make these decisions? Polaris sends an **Input Document** to OPA describing the *actor*, *action*, and *resource*. OPA evaluates the JSON document against the Rego policy and returns a boolean `allow`.

In [7]:
opa_input = {
  "input": {
    "actor": {
      "principal": "polaris-analyst",
      "roles": ["ANALYST"]
    },
    "action": "CREATE_TABLE_DIRECT",
    "resource": {
      "targets": [{
          "type": "TABLE",
          "name": "forecasts",
          "parents": [{"type": "NAMESPACE", "name": "finance"}, {"type": "CATALOG", "name": "opa_catalog"}]
      }]
    }
  }
}

r_opa = requests.post(f"{OPA_URL}/v1/data/polaris/authz/allow", json=opa_input)
print("Polaris asks OPA: Can ANALYST perform CREATE_TABLE_DIRECT?")
print("OPA Responds:\n", json.dumps(r_opa.json(), indent=2))

Polaris asks OPA: Can ANALYST perform CREATE_TABLE_DIRECT?
OPA Responds:
 {
  "decision_id": "b5e34269-451a-49bd-a143-9873caf33e55",
  "result": false
}


---
## 🧹 Cleanup
Clean up the test state using the admin token.

In [8]:
requests.delete(f"{POLARIS_URL}/api/catalog/v1/opa_catalog/namespaces/finance/tables/transactions", headers=admin_headers)
requests.delete(f"{POLARIS_URL}/api/catalog/v1/opa_catalog/namespaces/finance", headers=admin_headers)
requests.delete(f"{POLARIS_URL}/api/management/v1/catalogs/opa_catalog", headers=admin_headers)

for p in ["polaris-admin", "polaris-data-engineer", "polaris-analyst"]:
    requests.delete(f"{POLARIS_URL}/api/management/v1/principals/{p}", headers=root_headers)
    
for r in ["ADMIN", "DATA_ENGINEER", "ANALYST"]:
    # Note: principal roles are global, and typically kept around unless thoroughly unused, 
    # for our cleanup we will just delete them.
    requests.delete(f"{POLARIS_URL}/api/management/v1/principal-roles/{r}", headers=root_headers)

print("🗑️ Demonstration state cleaned up!")

🗑️ Demonstration state cleaned up!
